# features와 isFraud 관계 조사

train만 사용하여 지표 계산 진행 

In [25]:
import duckdb
import pandas as pd

from ieee_cis.config import WAREHOUSE_PATH

con = duckdb.connect(str(WAREHOUSE_PATH), read_only=True)
con.execute("SET memory_limit='3GB'")

BASE_RATE = con.execute(
    "SELECT AVG(isFraud) FROM txn WHERE dataset_split='train'"
).fetchone()[0]
print(f"기준 사기율 {BASE_RATE*100:.3f}%")

cols = [r[0] for r in con.execute("DESCRIBE txn").fetchall()]
dtypes = {r[0]: r[1] for r in con.execute("DESCRIBE txn").fetchall()}
len(cols)

기준 사기율 3.499%


438

## 1. 결측치를 어떻게 판단할 것인가

결측치가 많은 컬럼이 다수 존재. 어떻게 해석해야할지 판단해야할 필요성 있음.

In [26]:
EXCLUDE = {"TransactionID", "isFraud", "dataset_split", "TransactionDT"}
targets = [c for c in cols if c not in EXCLUDE]

parts = [
    f'AVG(CASE WHEN "{c}" IS NULL THEN isFraud END) AS null_{i}, '
    f'AVG(CASE WHEN "{c}" IS NOT NULL THEN isFraud END) AS notnull_{i}, '
    f'AVG(("{c}" IS NULL)::TINYINT) AS nullrate_{i}'
    for i, c in enumerate(targets)
]

row = con.execute(
    "SELECT " + ", ".join(parts) + " FROM txn WHERE dataset_split='train'"
).fetchone()

null_signal = pd.DataFrame(
    [
        {
            "column": c,
            "null_rate": row[i * 3 + 2],
            "fraud_when_null": row[i * 3],
            "fraud_when_present": row[i * 3 + 1],
        }
        for i, c in enumerate(targets)
    ]
)

# 결측이 있는 게 위험
# 결측이 아예 없거나 전부인 컬럼은 비교 불가
null_signal = null_signal[null_signal.null_rate.between(0.01, 0.99)].copy()
null_signal["lift"] = null_signal.fraud_when_null / null_signal.fraud_when_present
null_signal.sort_values("lift", ascending=False).head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column,null_rate,fraud_when_null,fraud_when_present,lift
8,addr1,0.111264,0.117813,0.024621,4.785026
9,addr2,0.111264,0.117813,0.024621,4.785026
48,M6,0.286788,0.070684,0.020637,3.425055
44,M2,0.459071,0.052826,0.019853,2.660767
45,M3,0.459071,0.052826,0.019853,2.660767
43,M1,0.459071,0.052826,0.019853,2.660767
53,V2,0.472935,0.052122,0.019617,2.656917
52,V1,0.472935,0.052122,0.019617,2.656917
61,V10,0.472935,0.052122,0.019617,2.656917
60,V9,0.472935,0.052122,0.019617,2.656917


In [27]:
# 결측이 없는 게 위험 (값이 있는 게 위험)
null_signal.sort_values("lift").head(10)

,column,null_rate,fraud_when_null,fraud_when_present,lift
34,D7,0.934099,0.026962,0.148778,0.181225
39,D12,0.890410,0.024847,0.117403,0.211636
41,D14,0.894695,0.025456,0.115989,0.219473
40,D13,0.895093,0.026156,0.110360,0.237011
33,D6,0.876068,0.025022,0.105456,0.237271
400,id_09,0.873123,0.024895,0.104463,0.238310
35,D8,0.873123,0.024895,0.104463,0.238310
401,id_10,0.873123,0.024895,0.104463,0.238310
36,D9,0.873123,0.024895,0.104463,0.238310
394,id_03,0.887689,0.025850,0.107231,0.241068


## 2. 수치형 컬럼 - 사기/정상 그룹 간 분포 차이

평균 차이를 표준편차로 나눈 표준화 효과크기(Cohen's d)로 순위를 매김.
스케일이 다른 컬럼을 척도로 비교하기 위해서 사용

참고: Cohen's d(코헨의 d)는 두 집단 평균의 차이가 얼마나 큰지 나타내는 효과 크기(Effect Size) 지표

In [28]:
NUMERIC = {"UTINYINT", "SMALLINT", "INTEGER", "BIGINT", "DOUBLE"}
numeric_cols = [c for c in targets if dtypes[c] in NUMERIC]

parts = []
for i, c in enumerate(numeric_cols):
    parts += [
        f'AVG(CASE WHEN isFraud=1 THEN "{c}" END) AS f_{i}',
        f'AVG(CASE WHEN isFraud=0 THEN "{c}" END) AS n_{i}',
        f'STDDEV("{c}") AS s_{i}',
    ]
row = con.execute(
    "SELECT " + ", ".join(parts) + " FROM txn WHERE dataset_split='train'"
).fetchone()

rows = []

for i, c in enumerate(numeric_cols):
    fm, nm, sd = row[i * 3], row[i * 3 + 1], row[i * 3 + 2]
    if fm is None or nm is None or not sd:
        continue
    rows.append({
        "column": c,
        "mean_fraud": fm,
        "mean_normal": nm,
        "cohens_d": (fm - nm) / sd,
    })

numeric_signal = pd.DataFrame(rows)
numeric_signal["abs_d"] = numeric_signal.cohens_d.abs()
numeric_signal.sort_values("abs_d", ascending=False).head(25)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column,mean_fraud,mean_normal,cohens_d,abs_d
82,V45,2.207936,1.081869,1.543036,1.543036
294,V257,2.961342,1.106015,1.427223,1.427223
81,V44,1.963548,1.052407,1.425565,1.425565
123,V86,1.640020,1.045453,1.416288,1.416288
124,V87,1.799658,1.075798,1.415778,1.415778
283,V246,2.494702,1.072598,1.366931,1.366931
281,V244,1.992543,1.044478,1.356690,1.356690
195,V158,1.779178,0.789374,1.347891,1.347891
279,V242,1.931024,1.044162,1.343506,1.343506
193,V156,1.604658,0.738923,1.337642,1.337642


## 3. 범주형 컬럼 - 카테고리 간 사기율 편차

최대/최소 사기율 비율 확인

In [29]:
CATEGORICAL = {"VARCHAR", "BOOLEAN"}
cat_cols = [c for c in targets if dtypes[c] in CATEGORICAL]

rows = []
for c in cat_cols:
    df = con.execute(f"""
        SELECT "{c}" AS value, COUNT(*) AS n, AVG(isFraud) AS rate
        FROM txn WHERE dataset_split='train' AND "{c}" IS NOT NULL
        GROUP BY 1 HAVING COUNT(*) >= 500
    """).df()
    if len(df) < 2:
        continue
    rows.append({
        "column": c,
        "n_categories": len(df),
        "min_rate": df.rate.min(),
        "max_rate": df.rate.max(),
        "spread": df.rate.max() / max(df.rate.min(), 1e-6),
    })
    
cat_signal = pd.DataFrame(rows).sort_values("spread", ascending=False)
cat_signal

,column,n_categories,min_rate,max_rate,spread
4,R_emaildomain,15,0.000000,0.165138,165137.614679
21,id_31,31,0.001546,0.218425,141.320951
3,P_emaildomain,24,0.003012,0.189624,62.955277
22,id_33,20,0.007507,0.137329,18.294223
20,id_30,26,0.012351,0.121667,9.850805
29,DeviceInfo,7,0.012903,0.109290,8.469945
0,ProductCD,5,0.020399,0.116873,5.729225
7,M4,3,0.027051,0.113739,4.204611
13,has_identity,2,0.020939,0.078470,3.747654
2,card6,2,0.024263,0.066785,2.752592


## 4. 파생 피처 후보 검증


In [30]:
con.execute("""
    SELECT CASE
            WHEN TransactionAmt = FLOOR(TransactionAmt) THEN '0자리(정수)'
            WHEN ROUND(TransactionAmt, 2) = TransactionAmt THEN '2자리'
            ELSE '3자리 이상'
        END AS amt_decimals,
        COUNT(*) AS n,
        ROUND(AVG(isFraud) * 100, 3) AS fraud_pct
    FROM txn WHERE dataset_split='train'
    GROUP BY 1 ORDER BY fraud_pct DESC
""").df()

,amt_decimals,n,fraud_pct
0,3자리 이상,61933,11.721
1,0자리(정수),305013,3.568
2,2자리,223594,1.128


In [31]:
# card1 빈도 인코딩 - 희귀카드 확인
con.execute("""
    WITH freq AS (
        SELECT card1, COUNT(*) AS cnt FROM txn GROUP BY 1
    )
    SELECT CASE
             WHEN cnt = 1 THEN 'a 1건'
             WHEN cnt <= 5 THEN 'b 2-5건'
             WHEN cnt <= 50 THEN 'c 6-50건'
             WHEN cnt <= 500 THEN 'd 51-500건'
             ELSE 'e 500건+'
            END AS card_freq,
            COUNT(*) AS n,
            ROUND(AVG(t.isFraud) * 100, 3) AS fraud_pct
    FROM txn t JOIN freq USING (card1)
    WHERE dataset_split='train'
    GROUP BY 1 ORDER BY 1
""").df()

,card_freq,n,fraud_pct
0,a 1건,2053,5.212
1,b 2-5건,7955,3.394
2,c 6-50건,56821,2.533
3,d 51-500건,129554,3.378
4,e 500건+,394157,3.671


In [32]:
# 카드별 직전 거래 경과시간
con.execute("""
    WITH gaps AS (
        SELECT isFraud,
                TransactionDT - LAG(TransactionDT) OVER (
                    PARTITION BY card1 ORDER BY TransactionDT
                ) AS gap_sec
        FROM txn WHERE dataset_split='train'
    )
    SELECT CASE
             WHEN gap_sec IS NULL THEN 'a 첫거래'
             WHEN gap_sec < 60 THEN 'b <1분'
             WHEN gap_sec < 3600 THEN 'c 1분-1시간'
             WHEN gap_sec < 86400 THEN 'd 1시간-1일'
             ELSE 'e 1일+'
           END AS gap_band,
           COUNT(*) AS n,
           ROUND(AVG(isFraud) * 100, 3) AS fraud_pct
    FROM gaps GROUP BY 1 ORDER BY 1
""").df()

,gap_band,n,fraud_pct
0,a 첫거래,13553,2.568
1,b <1분,19399,6.382
2,c 1분-1시간,227854,3.999
3,d 1시간-1일,215681,3.349
4,e 1일+,114053,2.402


## 5. V 컬럼 상관 관계 확인


In [33]:
# 상위 신호 V 컬럼끼리의 상관 확인
top_v = (
    numeric_signal[numeric_signal.column.str.match(r"^V\d+$")]
    .sort_values("abs_d", ascending=False)
    .head(30)
    .column.tolist()
)
col_list = ", ".join(f'"{c}"' for c in top_v)     # f-string 밖으로 분리
corr = con.execute(
    f"SELECT {col_list} FROM txn "
    f"WHERE dataset_split='train' USING SAMPLE 100000"
).df().corr()


# 0.95 이상 상관 쌍
pairs = [
    (a, b, corr.loc[a, b])
    for i, a in enumerate(top_v)
    for b in top_v[i + 1 :]
    if abs(corr.loc[a, b]) > 0.95
]
print(f"상관 0.95 이상인 쌍: {len(pairs)} 개")
pairs[:15]

상관 0.95 이상인 쌍: 11 개


[('V244', 'V242', np.float64(0.9777645327152913)),
 ('V158', 'V157', np.float64(0.9756043296731179)),
 ('V156', 'V149', np.float64(0.9806375578919079)),
 ('V156', 'V155', np.float64(0.9804856472916595)),
 ('V156', 'V148', np.float64(0.9642940179531324)),
 ('V149', 'V155', np.float64(0.9557739881073183)),
 ('V149', 'V148', np.float64(0.975047101866828)),
 ('V51', 'V94', np.float64(0.9524696475047939)),
 ('V40', 'V39', np.float64(0.952364517975349)),
 ('V155', 'V148', np.float64(0.9817504790764223)),
 ('V43', 'V42', np.float64(0.9514682037424124))]

## 6. 종합 - 상위 신호 

In [36]:
def add_rank(df, score_col, kind):
    out = df[["column"]].copy()
    out["kind"] = kind
    out["raw"] = df[score_col].values
    out["rank_pct"] = df[score_col].rank(pct=True).values
    return out

summary = pd.concat([
    add_rank(numeric_signal, "abs_d", "numeric"),
    add_rank(cat_signal[cat_signal.min_rate > 0], "spread", "categorical"),
    add_rank(null_signal.assign(nl=null_signal.lift.apply(lambda x: max(x, 1/x))),
             "nl", "null_pattern"),
])
summary.sort_values("rank_pct", ascending=False).head(40)


,column,kind,raw,rank_pct
21,id_31,categorical,141.320951,1.000000
34,D7,null_pattern,5.518017,1.000000
82,V45,numeric,1.543036,1.000000
294,V257,numeric,1.427223,0.997506
8,addr1,null_pattern,4.785026,0.995223
9,addr2,null_pattern,4.785026,0.995223
81,V44,numeric,1.425565,0.995012
123,V86,numeric,1.416288,0.992519
39,D12,null_pattern,4.725101,0.990446
124,V87,numeric,1.415778,0.990025


In [35]:
con.close()